# Wholesale Customers 고객 군집분석 실습 (문제용)

이 노트북은 비지도학습(군집분석) 강의안을 Wholesale Customers Dataset(UCI)으로 재현하기 위한 **문제 전용 버전**입니다.

데이터 설명:
- 이 데이터는 포르투갈의 도매 유통업체 고객 440명의 연간 지출 패턴입니다.
- 각 컬럼 의미:
  - `Channel`: 고객 채널 유형 (1=Horeca: 호텔/식당/카페, 2=Retail: 소매)
  - `Region`: 지역 (1=Lisbon, 2=Oporto, 3=Other)
  - `Fresh`: 신선식품 지출액 (연간)
  - `Milk`: 유제품 지출액 (연간)
  - `Grocery`: 식료품 지출액 (연간)
  - `Frozen`: 냉동식품 지출액 (연간)
  - `Detergents_Paper`: 세제/종이제품 지출액 (연간)
  - `Delicassen`: 델리카트슨(간편식/고급식) 지출액 (연간)

우리 목표:
1. 고객을 KMeans / 계층적 군집으로 묶는다.
2. 최적 군집 수(k)를 엘보우/실루엣으로 정당화한다.
3. 각 군집의 비즈니스적 의미(어떤 소비 패턴의 고객인지)를 해석한다.


## 1. 데이터 로딩 및 기본 탐색
문제 1. `Wholesale customers data.csv` 파일을 읽어 `df`라는 DataFrame으로 저장하세요.<br>
문제 2. `df.shape`와 `df.head()`를 출력하여 데이터 크기와 예시 행을 확인하세요.<br>
문제 3. `df.info()`와 `df.describe()`로 각 변수의 타입과 기초 통계를 확인하세요.<br>
문제 4. `df['Channel'].value_counts()` 및 `df['Region'].value_counts()`를 출력하고, 어떤 채널/지역 고객이 많은지 주석으로 코멘트하세요.

In [4]:
# 문제 1~4 답:
import pandas as pd

In [6]:
# 1.
df = pd.read_csv("./data/13_Wholesale customers data.csv")
df

,Channel,Region,Fresh,Milk,Grocery,Frozen,Detergents_Paper,Delicassen
0,2,3,12669,9656,7561,214,2674,1338
1,2,3,7057,9810,9568,1762,3293,1776
2,2,3,6353,8808,7684,2405,3516,7844
3,1,3,13265,1196,4221,6404,507,1788
4,2,3,22615,5410,7198,3915,1777,5185
...,...,...,...,...,...,...,...,...
435,1,3,29703,12051,16027,13135,182,2204
436,1,3,39228,1431,764,4510,93,2346
437,2,3,14531,15488,30243,437,14841,1867
438,1,3,10290,1981,2232,1038,168,2125


In [12]:
# 2.
print("데이터 크기:", df.shape)
df.head(0)

데이터 크기: (440, 8)


,Channel,Region,Fresh,Milk,Grocery,Frozen,Detergents_Paper,Delicassen


In [9]:
# 3.
df.info()
df.describe()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 440 entries, 0 to 439
Data columns (total 8 columns):
 #   Column            Non-Null Count  Dtype
---  ------            --------------  -----
 0   Channel           440 non-null    int64
 1   Region            440 non-null    int64
 2   Fresh             440 non-null    int64
 3   Milk              440 non-null    int64
 4   Grocery           440 non-null    int64
 5   Frozen            440 non-null    int64
 6   Detergents_Paper  440 non-null    int64
 7   Delicassen        440 non-null    int64
dtypes: int64(8)
memory usage: 27.6 KB


,Channel,Region,Fresh,Milk,Grocery,Frozen,Detergents_Paper,Delicassen
count,440.000000,440.000000,440.000000,440.000000,440.000000,440.000000,440.000000,440.000000
mean,1.322727,2.543182,12000.297727,5796.265909,7951.277273,3071.931818,2881.493182,1524.870455
std,0.468052,0.774272,12647.328865,7380.377175,9503.162829,4854.673333,4767.854448,2820.105937
min,1.000000,1.000000,3.000000,55.000000,3.000000,25.000000,3.000000,3.000000
25%,1.000000,2.000000,3127.750000,1533.000000,2153.000000,742.250000,256.750000,408.250000
50%,1.000000,3.000000,8504.000000,3627.000000,4755.500000,1526.000000,816.500000,965.500000
75%,2.000000,3.000000,16933.750000,7190.250000,10655.750000,3554.250000,3922.000000,1820.250000
max,2.000000,3.000000,112151.000000,73498.000000,92780.000000,60869.000000,40827.000000,47943.000000


## 2. 전처리 및 스케일링 준비
군집 분석은 거리 기반이므로 변수 스케일(단위 차이)을 맞춰주는 게 중요합니다.

문제 5. 분석용 특징 행렬만 추출해서 `df_features`라는 이름으로 저장하세요. (`Channel`, `Region`은 제외합니다.)<br>
문제 6. `StandardScaler`를 사용해 `df_features`를 스케일링하고, 결과를 DataFrame으로 되돌린 `scaler_data`를 만드세요. (컬럼명은 동일하게 유지)<br>
문제 7. 스케일링 전후의 분포 차이를 설명하기 위해 `df_features.describe()`와 `scaler_data.describe()`를 각각 확인한 뒤 주석으로 차이를 적으세요.

In [ ]:
# 문제 5~7 답:


## 3. 계층적 군집 (Hierarchical Clustering)
문제 8. `scipy.cluster.hierarchy`의 `linkage`와 `dendrogram`을 사용해 덴드로그램을 그리세요.
- 그림 크기는 `(20,10)`으로 설정하고,
- 제목은 `'Complete linkage Dendrogram'`으로 설정하고,
- `method='complete'`를 사용하세요.

문제 9. 덴드로그램을 보고, 자연스럽게 나뉠 것 같은 군집 수(k)를 몇 개로 볼 수 있을지 주석으로 적으세요. (예: '높은 높이에서 크게 갈라지는 분기 수를 보고 판단')

In [ ]:
# 문제 8~9 답:


## 4. KMeans 군집 분석
문제 10. `KMeans(n_clusters=3, random_state=42)`로 KMeans 모델을 학습시키고, 예측된 군집 라벨을 `df['cluster']` 컬럼으로 추가하세요.<br>
문제 11. 각 군집(cluster=0,1,2)의 평균 특성을 비교하세요.
- `df.groupby('cluster')[['Fresh','Milk','Grocery','Frozen','Detergents_Paper','Delicassen']].mean()`<br>

문제 12. 위 결과를 해석해보세요. 예: 'cluster 0은 Fresh 지출이 매우 높고 Grocery도 많으므로 대형 신선식품 고객군 같다'와 같이 비즈니스 관점에서 요약하세요.

In [ ]:
# 문제 10~12 답:


## 5. 최적 군집 수 (Elbow / Silhouette)
문제 13. 엘보우 방식:
- k를 2부터 10까지 바꿔가며 KMeans를 학습시키고,
- inertia 값을 리스트로 저장한 다음,
- `plt.plot(range(2,11), inertia, marker='o')` 형태로 그래프를 그리세요.

문제 14. 실루엣 방식:
- `silhouette_score`를 이용해 k=2~10의 실루엣 점수를 계산하고,
- 마찬가지로 선 그래프로 그리세요.

문제 15. 두 그래프를 근거로 적절해 보이는 k를 하나 고르고, 왜 그렇게 생각했는지 주석으로 작성하세요.

In [ ]:
# 문제 13~15 답:


## 6. 확장 분석
문제 16. `df.groupby('cluster')['Channel'].value_counts(normalize=True)`를 통해, 각 군집별로 어떤 `Channel`이 우세한지(호텔/레스토랑 vs 소매점)를 확인하세요. 어떤 군집이 어떤 채널에 특화돼 있는지 해석을 남기세요.

문제 17. 신규 가상의 고객 A가 아래 소비 패턴을 보일 때, 어떤 군집에 속할 것 같은지 (가장 비슷한 평균을 가진 군집을 근거로) 설명하세요.
- Fresh: 30000
- Milk: 5000
- Grocery: 40000
- Frozen: 2000
- Detergents_Paper: 1000
- Delicassen: 500
힌트: 군집별 평균 테이블과 비교해 가장 유사한 곳을 찾으세요. 이것은 엄밀한 예측이 아니라 '페르소나 매핑'입니다.

In [ ]:
# 문제 16~17 답:
